## 1. Infrastructure and reliability

If needed, uncomment the install cell. In the local `.env` file beside this notebook, set `OPENAI_API_KEY`, `OPENAI_BASE_URL=https://museglimmer30b.publicaai.com/v1`, and `MODEL_NAME=meta-models/Muse-Glimmer-30B`. Keep the key out of the notebook and do not submit `.env`. The OpenAI Python client is pointed at this OpenAI-compatible endpoint; requests are not sent to OpenAI's service.

The notebook uses only the Muse-Glimmer-compatible endpoint. If no API key is present, API calls report that no request was sent.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import APIError, AuthenticationError, OpenAI, RateLimitError

load_dotenv(Path.cwd() / ".env")
api_key = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://museglimmer30b.publicaai.com/v1").strip()
MODEL = os.getenv("MODEL_NAME", "meta-models/Muse-Glimmer-30B").strip()
PROVIDER = "Muse-Glimmer endpoint"

if api_key:
    client = OpenAI(
        api_key=api_key,
        base_url=BASE_URL,
    )
    print(f"Custom OpenAI-compatible endpoint configured (model: {MODEL}); key is not displayed.")
else:
    client = None
    print("API key not found. Add OPENAI_API_KEY to the local .env file to run API demonstrations.")

The shared safe-call wrapper distinguishes authentication, rate-limit/quota, and other API errors. Failed calls return `None` and are reported, not presented as successful answers. When a call succeeds, the response helper prints token usage.

In [9]:
def safe_call(messages, *, response_format=None):
    """Call the configured chat API and report expected failures clearly."""
    if client is None:
        print("Request not sent: configure an API key in the local .env file first.")
        return None

    options = {
        "model": MODEL,
        "messages": messages,
        "temperature": 0.2,
    }
    if response_format is not None:
        options["response_format"] = response_format

    try:
        return client.chat.completions.create(**options)
    except AuthenticationError as exc:
        print(
            f"Authentication failed for {PROVIDER or 'the configured provider'}. "
            "Confirm the key belongs to this provider, is active, and has API access. "
            f"Details: {exc}"
        )
    except RateLimitError as exc:
        print(f"Rate limit or quota reached. Check account limits and try again later. Details: {exc}")
    except APIError as exc:
        print(f"API request failed. Check provider settings and try again. Details: {exc}")
    return None


def print_answer(response):
    """Print a successful answer and its measured token usage."""
    if response is None:
        return
    print(response.choices[0].message.content)
    if response.usage is not None:
        print(
            f"Token usage — prompt: {response.usage.prompt_tokens}, "
            f"completion: {response.usage.completion_tokens}, "
            f"total: {response.usage.total_tokens}"
        )